<a href="https://colab.research.google.com/github/huggingface/deep-rl-class/blob/main/notebooks/bonus-unit1/bonus_unit1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bonus Unit 1:训练机器狗 Huggy 🐶 去捡棍子

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit2/thumbnail.png" alt="Bonus Unit 1Thumbnail">

在本 notebook 中,我们将通过**教机器狗 Huggy 去捡棍子,然后直接在你的浏览器里和它一起玩**,来巩固第一个单元中学到的内容

⬇️ 下面是一个示例,展示了**你在本单元结束时能够实现的效果** ⬇️(点击 ▶ 播放)

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.mp4" type="video/mp4"></video>

### 环境 🎮

- 机器狗 Huggy,一个由 [Thomas Simonini](https://twitter.com/ThomasSimonini) 基于 [Puppo The Corgi](https://blog.unity.com/technology/puppo-the-corgi-cuteness-overload-with-the-unity-ml-agents-toolkit) 创建的环境

### 使用的库 📚

- [MLAgents](https://github.com/Unity-Technologies/ml-agents)

我们一直在努力改进教程,所以**如果你在本 notebook 中发现了问题**,请[在 GitHub 仓库上提一个 issue](https://github.com/huggingface/deep-rl-class/issues)。

## 本 notebook 的目标 🏆

完成本 notebook 后,你将:

- 理解**用于训练 Huggy 的状态空间、动作空间和奖励函数**。
- **训练你自己的 Huggy** 去捡棍子。
- 能够**在浏览器中直接和你训练好的 Huggy 一起玩**。


## 本 notebook 来自深度强化学习课程
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

在这门免费课程中,你将:

- 📖 从**理论和实践**两方面学习深度强化学习。
- 🧑‍💻 学会**使用知名的深度强化学习库**,例如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在**独特的环境中训练智能体**

想了解更多,请查看 📚 课程大纲 👉 https://simoninithomas.github.io/deep-rl-course

别忘了**<a href="http://eepurl.com/ic5ZUD">注册这门课程</a>**(我们收集你的邮箱是为了**在每个单元发布时把链接发送给你,并向你提供有关挑战和更新的信息。**)


保持联系的最佳方式是加入我们的 Discord 服务器,与社区以及我们交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 前置要求 🏗️

在深入学习本 notebook 之前,你需要:

🔲 📚 通过学习第 1 单元,**理解强化学习的基础知识**(MC、TD、奖励假设等)

🔲 📚 通过学习 Bonus Unit 1,**阅读 Huggy 的介绍**

## 设置 GPU 💪
- 为了**加速智能体的训练,我们将使用 GPU**。为此,请进入 `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## 克隆仓库 🔽

- 我们需要克隆包含 **ML-Agents** 的仓库。

In [ ]:
%%capture
# 克隆仓库(可能需要 3 分钟)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

## 设置虚拟环境 🔽
- 要让 **ML-Agents** 在 Colab 中成功运行,Colab 的 Python 版本必须满足该库对 Python 的要求。

- 我们可以在 `setup.py` 文件中的 `python_requires` 参数下查看支持的 Python 版本。这些文件是安装使用 **ML-Agents** 库所必需的,位于以下位置:
  - `/content/ml-agents/ml-agents/setup.py`
  - `/content/ml-agents/ml-agents-envs/setup.py`

- Colab 当前的 Python 版本(可以用 `!python --version` 查看)与该库的 `python_requires` 参数不匹配,因此安装可能会静默失败,导致之后执行相同命令时出现如下错误:
  - `/bin/bash: line 1: mlagents-learn: command not found`
  - `/bin/bash: line 1: mlagents-push-to-hf: command not found`

- 为了解决这个问题,我们将创建一个 Python 版本与 **ML-Agents** 库兼容的虚拟环境。

`注意:` *为了后续的兼容性,请务必检查安装文件中的 `python_requires` 参数;如果 Colab 的 Python 版本不兼容,请在下面给出的脚本中把虚拟环境设置为所支持的最高 Python 版本*

In [ ]:
# Colab 当前的 Python 版本(与 ML-Agents 不兼容)
!python --version

In [ ]:
# 安装 virtualenv 并创建一个虚拟环境
!pip install virtualenv
!virtualenv myenv

# 下载并安装 Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# 激活 Miniconda 并安装 Python 3.10.12 版本
!source /usr/local/bin/activate
!conda install -q -y --prefix /usr/local python=3.10.12 ujson  # 在此处指定版本

# 为 Python 和 conda 路径设置环境变量
!export PYTHONPATH=/usr/local/lib/python3.10/site-packages/
!export CONDA_PREFIX=/usr/local/envs/myenv

In [ ]:
# 新虚拟环境中的 Python 版本(与 ML-Agents 兼容)
!python --version

## 安装依赖 🔽

In [ ]:
%%capture
# 进入仓库并安装软件包(可能需要 3 分钟)
%cd ml-agents
!pip3 install -e ./ml-agents-envs
!pip3 install -e ./ml-agents

## 下载环境 zip 文件并将其移动到 `./trained-envs-executables/linux/`

- 我们的环境可执行文件在一个 zip 文件里。
- 我们需要下载它,并把它放到 `./trained-envs-executables/linux/`

In [ ]:
!mkdir ./trained-envs-executables
!mkdir ./trained-envs-executables/linux

我们使用 `wget` 从 https://github.com/huggingface/Huggy 下载了文件 Huggy.zip

In [ ]:
!wget "https://github.com/huggingface/Huggy/raw/main/Huggy.zip" -O ./trained-envs-executables/linux/Huggy.zip

In [ ]:
%%capture
!unzip -d ./trained-envs-executables/linux/ ./trained-envs-executables/linux/Huggy.zip

请确保你的文件可以访问

In [ ]:
!chmod -R 755 ./trained-envs-executables/linux/Huggy

## 我们来回顾一下这个环境是如何工作的

### 状态空间:Huggy「感知」到了什么

Huggy 并不能「看到」它的环境,而是由我们向它提供关于环境的信息:

- 目标(棍子)的位置
- 它自身与目标之间的相对位置
- 它四条腿的朝向

凭借所有这些信息,Huggy **可以决定下一步采取什么动作来实现自己的目标**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.jpg" alt="Huggy" width="100%">


### 动作空间:Huggy 可以做哪些移动
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-action.jpg" alt="Huggy action" width="100%">

**关节马达驱动着 Huggy 的腿**。这意味着为了拿到目标,Huggy 需要**学会正确地转动每条腿的关节马达,才能移动起来**。

### 奖励函数

奖励函数的设计目标是让 **Huggy 实现自己的目标**:捡回棍子。

回忆一下,强化学习的基石之一是*奖励假设*:任何目标都可以被描述为**对期望累积奖励的最大化**。

在这里,我们的目标是让 Huggy **朝棍子跑去,同时不要原地打转太多**。因此,我们的奖励函数必须体现这一目标。

我们的奖励函数:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/reward.jpg" alt="Huggy reward function" width="100%">

- *朝向奖励*:我们**在它接近目标时给予奖励**。
- *时间惩罚*:每个动作都会附加一个固定的时间惩罚,**促使它尽快拿到棍子**。
- *旋转惩罚*:如果 Huggy **打转太多、转得太快**,我们就惩罚它。
- *到达目标奖励*:我们**在 Huggy 到达目标时给予奖励**。

## 创建 Huggy 的配置文件

- 在 ML-Agents 中,你需要在 config.yaml 文件中定义**训练超参数**。

- 在本 notebook 的范围内,我们不会修改超参数;但如果你想做个实验,也可以尝试修改其他一些超参数,Unity 在这里提供了非常[好的文档对它们一一进行说明](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md)。

- 但我们需要为 Huggy 创建一个配置文件。

  - 为此,请点击屏幕左侧的文件夹图标。

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/create_file.png" alt="Create file" width="10%">

  - 进入 `/content/ml-agents/config/ppo` 目录
  - 点击鼠标右键,新建一个名为 `Huggy.yaml` 的文件

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/create-huggy.png" alt="Create huggy.yaml" width="20%">

- 复制并粘贴下面的内容 🔽

In [ ]:
behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2e6
    time_horizon: 1000
    summary_freq: 50000

- 别忘了保存文件!

- **如果你想修改超参数**,在 Google Colab notebook 中,你可以点击这里打开 config.yaml:`/content/ml-agents/config/ppo/Huggy.yaml`

- 举例来说,**如果你想在训练过程中保存更多模型**(目前我们每 200,000 个训练时间步保存一次),你需要修改:
  - `checkpoint_interval`:每个检查点之间收集的训练时间步数量。
  - `keep_checkpoints`:保留的模型检查点最大数量。

=> 只需记住:**减小 `checkpoint_interval` 意味着要上传到 Hub 的模型更多,因此上传时间也更长**
现在我们已经准备好训练智能体了 🔥。

## 训练我们的智能体

要训练智能体,我们只需要**启动 mlagents-learn 并指定包含环境的可执行文件。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/mllearn.png" alt="ml learn function" width="100%">

使用 ML-Agents 时,我们运行一个训练脚本,并定义四个参数:

1. `mlagents-learn <config>`:超参数配置文件所在的路径。
2. `--env`:环境可执行文件所在的位置。
3. `--run-id`:你想给这次训练 run id 起的名字。
4. `--no-graphics`:训练期间不启动可视化界面。

训练模型,并使用 `--resume` 标志在中断后继续训练。

> 第一次使用 `--resume` 时会失败,再次运行该单元格即可绕过这个错误。


根据你的机器不同,训练将耗时 30 到 45 分钟(别忘了**设置 GPU**),去喝杯 ☕️ 吧,这是你应得的 🤗。

In [ ]:
!mlagents-learn ./config/ppo/Huggy.yaml --env=./trained-envs-executables/linux/Huggy/Huggy --run-id="Huggy2" --no-graphics

## 把智能体推送到 🤗 Hub

- 既然我们已经训练好了智能体,接下来就可以**把它推送到 Hub,从而在浏览器里和 Huggy 一起玩 🔥。**

要与社区分享你的模型,还需要再完成三个步骤:

1️⃣ (如果还没有的话)注册一个 HF 账号 ➡ https://huggingface.co/join

2️⃣ 登录之后,你需要保存来自 Hugging Face 网站的认证令牌(token)。
- 创建一个新令牌(https://huggingface.co/settings/tokens),**权限要选择写入(write)**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- 复制该令牌
- 运行下面的单元格并粘贴令牌

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

如果你不想使用 Google Colab 或 Jupyter Notebook,则需要改用这条命令:`huggingface-cli login`

然后,我们只需要运行 `mlagents-push-to-hf`。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/mlpush.png" alt="ml learn function" width="100%">

这里我们定义 4 个参数:

1. `--run-id`:训练 run id 的名字。
2. `--local-dir`:智能体保存的位置,即 results/<run_id 名字>,以我为例就是 results/First Training。
3. `--repo-id`:你想创建或更新的 Hugging Face 仓库名,形式始终为 <你的 Hugging Face 用户名>/<仓库名>。
如果该仓库不存在,**它会被自动创建**。
4. `--commit-message`:由于 HF 仓库是 git 仓库,你需要指定一条提交信息。

In [ ]:
!mlagents-push-to-hf --run-id="HuggyTraining" --local-dir="./results/Huggy2" --repo-id="ThomasSimonini/ppo-Huggy" --commit-message="Huggy"

否则,如果一切顺利,你应该会在流程结束时看到如下内容(不过 URL 会不同 😆):

```
Your model is pushed to the hub. You can view your model here: https://huggingface.co/ThomasSimonini/ppo-Huggy
```

这是你的模型仓库的链接。该仓库包含一个模型卡(model card),说明了如何使用该模型,还有你的 Tensorboard 日志和配置文件。**最棒的是它是一个 git 仓库,这意味着你可以拥有不同的提交、通过新的推送更新仓库、发起 Pull Request 等等。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/modelcard.png" alt="ml learn function" width="100%">

现在来的是最精彩的部分:**在线和 Huggy 一起玩 👀。**

## 和你的 Huggy 一起玩 🐕

这一步最简单:

- 在浏览器中打开 Huggy 游戏:https://huggingface.co/spaces/ThomasSimonini/Huggy

- 点击「Play with my Huggy model」

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/load-huggy.jpg" alt="load-huggy" width="100%">

1. 在第 1 步中,选择你的模型仓库,即模型 id(以我为例就是 ThomasSimonini/ppo-Huggy)。

2. 在第 2 步中,**选择你想回放哪一个模型**:
  - 我这里有多个模型,因为我们每 500000 个时间步就保存一次模型。
  - 但由于我想要最新的那个,我选择了 `Huggy.onnx`

👉 值得一试的是**用不同训练阶段的模型来对比,观察智能体的进步。**

恭喜你完成了这个附加单元!

现在你可以坐下来,尽情地和你的 Huggy 一起玩了 🐶。别忘了**把这份爱传递下去,和你的朋友们分享 Huggy 🤗**。如果你在社交媒体上分享它,**请 @我们 @huggingface 和 @simoninithomas**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-cover.jpeg" alt="Huggy cover" width="100%">


## 持续学习,保持出色 🤗